# Ouro-1.4B — train BOTH Σ₀ adapters → HuggingFace (public)

Fine-tunes two QLoRA adapters on **ByteDance/Ouro-1.4B** and pushes both to
**https://huggingface.co/lanternfounder/ouro-checkpoints**:

- **`coding-adapter/`** — Σ₀ coding adapter (Claude-session coding data).
- **`capability-adapter/`** — coding + function-calling positives (Hermes) + irrelevance negatives (xLAM).

### Before you Run All
1. **Runtime → GPU.** Prefer an **Ampere+** GPU (Colab **L4/A100**, Lightning **A10**). The recipe uses **bf16**;
   free **T4/P100 are pre-Ampere (no bf16)** — the train script falls back to fp16, which can go **NaN** on this
   reasoning LM, so adapter quality there is not dependable.
2. **HF token (WRITE).** Colab: 🔑 *Secrets* → add `HF_TOKEN`. Kaggle: *Add-ons → Secrets* → `HF_TOKEN`.
   (Or set the `HF_TOKEN` env var.)
3. **Runtime → Run all.**


## 0 · Parameters


In [ ]:
HF_REPO   = "lanternfounder/ouro-checkpoints"   # public target
BASE      = "ByteDance/Ouro-1.4B"               # training base (not the -Thinking serve checkpoint)
SEQ       = 1536                                 # audited p99=1219 on the FC corpus
LORA_R    = 32                                   # alpha = 2*r
EPOCHS    = 3
MAX_STEPS = 600          # safety cap to fit a cloud session (~6h on an Ampere GPU @ ~38s/step); -1 = full epochs
REPO_URL  = "https://github.com/alex-place/lantern-os"


## 1 · Install dependencies
Pin `transformers>=4.57` — Ouro-1.4B's custom modeling code needs `layer_type_validation` (4.54+).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.57,<4.58", "peft>=0.10", "bitsandbytes>=0.43",
    "datasets", "accelerate", "scipy", "huggingface_hub", "zstandard"], check=True)
print("deps installed")


## 2 · Clone the repo (train script + datasets)


In [ ]:
import os, subprocess
REPO = "/kaggle/working/lantern-os" if os.path.isdir("/kaggle/working") else "/content/lantern-os"
if not os.path.isdir(REPO):
    env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}   # skip LFS blobs (budget) — we only need code + jsonl
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], env=env, check=True)
os.chdir(REPO)
print("repo at", REPO)


## 3 · HuggingFace login (WRITE token)


In [ ]:
import os
from huggingface_hub import login, HfApi
tok = os.environ.get("HF_TOKEN")
if not tok:
    try:
        from google.colab import userdata; tok = userdata.get("HF_TOKEN")
    except Exception: pass
if not tok:
    try:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
assert tok, "No HF_TOKEN found. Add it as a Colab/Kaggle secret or env var (needs WRITE access)."
login(token=tok)
api = HfApi(token=tok)
print("HF user:", api.whoami()["name"])


## 4 · GPU / precision check


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {name}  cc {cap[0]}.{cap[1]}")
    if cap[0] < 8:
        print("⚠️  pre-Ampere (cc<8.0): no bf16. The train script falls back to fp16 (NaN risk). "
              "Prefer Colab L4/A100 or Lightning A10 for a dependable adapter.")
else:
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU.")


## 5 · Build the two datasets (from committed files)
`coding` = the session coding set. `capability` = coding + Hermes FC positives + xLAM irrelevance negatives —
the exact 3 sources `retrain-combined.ps1` merges.


In [ ]:
import pathlib, json
M = "models/lantern-sigma0-coder"

def merge(srcs, out):
    n = 0
    with open(out, "w", encoding="utf-8") as f:
        for s in srcs:
            for line in pathlib.Path(s).read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                try: json.loads(line)
                except Exception: continue
                f.write(line + "\n"); n += 1
    print(f"{out}  rows: {n}")
    return out

coding_data   = f"{M}/training-data.jsonl"
capability_data = merge([f"{M}/training-data.jsonl", f"{M}/fc-hermes.jsonl", f"{M}/fc-negatives.jsonl"],
                        "/tmp/capability-training.jsonl")
print("coding rows:", sum(1 for _ in open(coding_data, encoding="utf-8")))


## 6 · Train helper


In [ ]:
import subprocess, sys, os
os.environ["HF_HOME"] = "/tmp/hf-cache"

def train(data, out, max_steps=MAX_STEPS):
    args = [sys.executable, "scripts/train-qlora-ouro.py",
            "--base", BASE, "--data", data, "--out", out,
            "--seq", str(SEQ), "--lora-r", str(LORA_R), "--epochs", str(EPOCHS)]
    if max_steps and int(max_steps) > 0:
        args += ["--max-steps", str(max_steps)]
    print("TRAIN:", " ".join(args))
    subprocess.run(args, check=True)
    final = os.path.join(out, "final")
    return final if os.path.isdir(final) else out


## 7 · Adapter 1 — coding → push


In [ ]:
from huggingface_hub import upload_folder
coding_adapter = train(coding_data, "/tmp/out-coding")
print("coding adapter at", coding_adapter)
upload_folder(folder_path=coding_adapter, path_in_repo="coding-adapter",
              repo_id=HF_REPO, repo_type="model",
              commit_message="coding adapter (Σ₀ sessions)")
print("✅ https://huggingface.co/%s/tree/main/coding-adapter" % HF_REPO)


## 8 · Adapter 2 — capability → push


In [ ]:
from huggingface_hub import upload_folder
capability_adapter = train(capability_data, "/tmp/out-capability")
print("capability adapter at", capability_adapter)
upload_folder(folder_path=capability_adapter, path_in_repo="capability-adapter",
              repo_id=HF_REPO, repo_type="model",
              commit_message="capability adapter (coding + FC + negatives)")
print("✅ https://huggingface.co/%s/tree/main/capability-adapter" % HF_REPO)


## 9 · Model card


In [ ]:
from huggingface_hub import upload_file
card = f"""---
license: apache-2.0
base_model: ByteDance/Ouro-1.4B
library_name: peft
tags: [lora, qlora, ouro, sigma0, lantern-os]
---
# Ouro-1.4B Σ₀ adapters

Two QLoRA adapters for `ByteDance/Ouro-1.4B` (Lantern / Keystone OS).

- `coding-adapter/` — Σ₀ coding adapter (Claude-session coding data).
- `capability-adapter/` — coding + function-calling positives (Hermes) + irrelevance negatives (xLAM).

**Recipe:** QLoRA r={LORA_R}/α={2*LORA_R}, seq {SEQ}, {EPOCHS} epochs, bf16.
See https://github.com/alex-place/lantern-os — `docs/SIGMA0-OURO-CODER.md`.

## Use
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = "ByteDance/Ouro-1.4B"
tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
m = AutoModelForCausalLM.from_pretrained(base, trust_remote_code=True, device_map="auto")
m = PeftModel.from_pretrained(m, "{HF_REPO}", subfolder="coding-adapter")
```
"""
open("/tmp/README.md", "w", encoding="utf-8").write(card)
upload_file(path_or_fileobj="/tmp/README.md", path_in_repo="README.md",
            repo_id=HF_REPO, repo_type="model", commit_message="model card")
print("✅ done → https://huggingface.co/%s" % HF_REPO)
